# Medical RAG Agent Quick Start

This notebook sets up the repo from a fresh notebook session, syncs dependencies with `uv`, and runs a small smoke test.

If you need access to a private GitHub repo or gated Hugging Face model, enter the tokens when prompted.

In [ ]:
import getpass
import os
import shutil
import subprocess
import sys
from pathlib import Path

if shutil.which("uv") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])

repo_url = os.environ.get("REPO_URL", "https://github.com/tsechinchi/medical-rag-agent.git")
github_token = os.environ.get("GITHUB_TOKEN", "").strip()
if not github_token:
    github_token = getpass.getpass("GitHub token (blank if not needed): ").strip()
    if github_token:
        os.environ["GITHUB_TOKEN"] = github_token

clone_url = repo_url
if github_token and repo_url.startswith("https://github.com/"):
    clone_url = repo_url.replace("https://", f"https://{github_token}@")

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    target = repo_root / "medical-rag-agent"
    if not target.exists():
        subprocess.check_call(["git", "clone", clone_url])
    repo_root = target

os.chdir(repo_root)
print(f"Working directory: {Path.cwd()}")

In [ ]:
hf_token = getpass.getpass("Hugging Face token (blank if not needed): ").strip()
if hf_token:
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    # Respect any pre-existing offline setting instead of forcing a hub call.
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "0")
print("Hugging Face token configured:" , bool(hf_token))

In [ ]:
!uv sync --extra gpu

## Download Model Once

This stores the model snapshot under `models/biomistral-7b/` so later runs can load from disk.

In [ ]:
from pathlib import Path

from src.model.loader import _download_model_snapshot
from config import config as app_config

cache_path = Path(app_config.MODEL_CACHE_DIR)
model_snapshot_ready = cache_path.exists()
if model_snapshot_ready:
    print(f'Model snapshot already available at {cache_path}.')
else:
    print(f'Model snapshot not found at {cache_path}; downloading from Hugging Face...')
    _download_model_snapshot(app_config.MODEL_ID, app_config.REVISION)
    model_snapshot_ready = True
    print('Model snapshot downloaded to local cache.')

## Smoke Test

This first run keeps the setup lightweight by skipping BERTScore.

In [ ]:
subprocess.check_call([sys.executable, "-m", "src.evaluation.run_eval", "--n_samples", "3", "--skip-bertscore"])